# Fused and Online Softmax Kernels in Triton

Two row-wise softmax GPU kernels written in Triton, verified against PyTorch and benchmarked:

1. **Fused softmax** – loads a whole row into on-chip memory, computes max / exp / sum / divide there, and writes once. One memory round-trip instead of ~5.
2. **Online (tiled) softmax** – for rows too wide to fit on-chip, streams the row in fixed-size tiles and keeps a *running* max and normalizer, rescaling the partial sum whenever the max changes (the same trick FlashAttention uses). Works for any row width with constant on-chip memory.

**Run:** Google Colab → Runtime → Change runtime type → **T4 GPU** → Run all.

In [ ]:
!pip -q install triton
import torch, triton, triton.language as tl
assert torch.cuda.is_available(), "Switch the Colab runtime to a GPU (Runtime > Change runtime type > T4 GPU)"
print(torch.cuda.get_device_name(0), "| torch", torch.__version__, "| triton", triton.__version__)

## 1. The kernel

Each program instance handles one row. The row is loaded into on-chip memory, padded to a power of two (`BLOCK_SIZE`), with out-of-range entries masked to `-inf` so they contribute nothing. Subtracting the row max before `exp` keeps the computation numerically stable.

In [ ]:
@triton.jit
def softmax_kernel(out_ptr, in_ptr, in_row_stride, out_row_stride, n_cols,
                   BLOCK_SIZE: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK_SIZE)
    mask = cols < n_cols

    x = tl.load(in_ptr + row * in_row_stride + cols, mask=mask, other=-float("inf"))
    x = x - tl.max(x, axis=0)          # numerical stability
    num = tl.exp(x)
    den = tl.sum(num, axis=0)
    tl.store(out_ptr + row * out_row_stride + cols, num / den, mask=mask)


def triton_softmax(x: torch.Tensor) -> torch.Tensor:
    n_rows, n_cols = x.shape
    BLOCK_SIZE = triton.next_power_of_2(n_cols)
    # more warps for wider rows so each row's reduction has enough threads
    num_warps = 4 if BLOCK_SIZE < 2048 else (8 if BLOCK_SIZE < 4096 else 16)
    y = torch.empty_like(x)
    softmax_kernel[(n_rows,)](y, x, x.stride(0), y.stride(0), n_cols,
                              BLOCK_SIZE=BLOCK_SIZE, num_warps=num_warps)
    return y


def naive_softmax(x: torch.Tensor) -> torch.Tensor:
    # unfused reference: each step is its own kernel and memory round-trip
    x_max = x.max(dim=1, keepdim=True)[0]
    z = x - x_max
    num = torch.exp(z)
    den = num.sum(dim=1, keepdim=True)
    return num / den

## 1b. Online (tiled) softmax for wide rows

The fused kernel needs `BLOCK_SIZE ≥ n_cols`, so very wide rows (vocabulary-sized, e.g. 50k–130k) overflow registers and spill. The online kernel instead walks the row in tiles of `TILE` columns:

* **Pass 1:** for each tile, update a running max `m` and running sum `s`. When the max grows from `m` to `m_new`, the previous sum is rescaled by `exp(m − m_new)` so it stays correct.
* **Pass 2:** re-read each tile and write `exp(x − m) / s`.

On-chip memory is fixed at one tile, regardless of row width.

In [ ]:
@triton.jit
def online_softmax_kernel(out_ptr, in_ptr, in_row_stride, out_row_stride, n_cols,
                          TILE: tl.constexpr):
    row = tl.program_id(0)
    in_row = in_ptr + row * in_row_stride
    out_row = out_ptr + row * out_row_stride
    offs = tl.arange(0, TILE)

    # pass 1: per-lane running max and running sum
    m = tl.full([TILE], float("-inf"), tl.float32)
    s = tl.zeros([TILE], tl.float32)
    for start in range(0, n_cols, TILE):
        cols = start + offs
        x = tl.load(in_row + cols, mask=cols < n_cols, other=float("-inf")).to(tl.float32)
        m_new = tl.maximum(m, x)
        # rescale the old sum to the new max; guard lanes that have only seen -inf
        alpha = tl.where(m_new == float("-inf"), 0.0, tl.exp(m - m_new))
        beta = tl.where(m_new == float("-inf"), 0.0, tl.exp(x - m_new))
        s = s * alpha + beta
        m = m_new
    # combine the per-lane statistics into one row max and normalizer
    row_max = tl.max(m, axis=0)
    row_sum = tl.sum(tl.where(m == float("-inf"), 0.0, s * tl.exp(m - row_max)), axis=0)

    # pass 2: normalize and write
    for start in range(0, n_cols, TILE):
        cols = start + offs
        mask = cols < n_cols
        x = tl.load(in_row + cols, mask=mask, other=float("-inf")).to(tl.float32)
        tl.store(out_row + cols, tl.exp(x - row_max) / row_sum, mask=mask)


def online_softmax(x: torch.Tensor, TILE: int = 1024) -> torch.Tensor:
    n_rows, n_cols = x.shape
    y = torch.empty_like(x)
    online_softmax_kernel[(n_rows,)](y, x, x.stride(0), y.stride(0), n_cols,
                                     TILE=TILE, num_warps=4)
    return y

## 2. Correctness

In [ ]:
torch.manual_seed(0)
shapes = [(1, 1), (7, 13), (1823, 781), (4096, 1000), (128, 8192), (64, 50257), (16, 131072)]
for shape in shapes:
    x = torch.randn(*shape, device="cuda") * 3
    ref = torch.softmax(x, dim=1)
    for name, fn in [("fused", triton_softmax), ("online", online_softmax)]:
        if name == "fused" and shape[1] > 16384:
            continue  # fused kernel is meant for rows that fit on-chip
        out = fn(x)
        err = (out - ref).abs().max().item()
        assert torch.allclose(out, ref, atol=1e-6, rtol=1e-5), (name, shape, err)
        print(f"{name:>6} {str(shape):>14}  max abs error {err:.2e}  OK")
# extreme values: very large / very negative logits must not overflow
x = torch.tensor([[1000., 999., -1000.], [-1e4, -1e4, -1e4]], device="cuda")
assert torch.allclose(online_softmax(x), torch.softmax(x, 1)) and torch.allclose(triton_softmax(x), torch.softmax(x, 1))
print("extreme-value test OK")

## 3. Benchmark

Throughput is reported as effective memory bandwidth (GB/s): the kernel reads the matrix once and writes it once, so bytes moved = 2 × rows × cols × 4. Higher is better; the T4's peak is about 320 GB/s.

In [ ]:
def gbps(ms, x):
    return 2 * x.numel() * x.element_size() / (ms * 1e-3) / 1e9

rows = 4096
results = []
for n_cols in [256, 512, 1024, 2048, 4096, 8192]:
    x = torch.randn(rows, n_cols, device="cuda")
    t_triton = triton.testing.do_bench(lambda: triton_softmax(x))
    t_torch  = triton.testing.do_bench(lambda: torch.softmax(x, dim=1))
    t_naive  = triton.testing.do_bench(lambda: naive_softmax(x))
    results.append((n_cols, gbps(t_triton, x), gbps(t_torch, x), gbps(t_naive, x),
                    t_naive / t_triton, t_torch / t_triton))

print(f"{'cols':>6} | {'Triton GB/s':>11} | {'torch GB/s':>10} | {'naive GB/s':>10} | {'vs naive':>8} | {'vs torch':>8}")
for c, tr, to, na, s_na, s_to in results:
    print(f"{c:>6} | {tr:>11.1f} | {to:>10.1f} | {na:>10.1f} | {s_na:>7.2f}x | {s_to:>7.2f}x")

In [ ]:
import matplotlib.pyplot as plt
cols = [r[0] for r in results]
plt.plot(cols, [r[1] for r in results], "o-", label="Triton fused")
plt.plot(cols, [r[2] for r in results], "s-", label="torch.softmax")
plt.plot(cols, [r[3] for r in results], "^-", label="naive (unfused)")
plt.xscale("log", base=2); plt.xlabel("columns (rows = 4096)"); plt.ylabel("GB/s")
plt.title(f"Softmax throughput on {torch.cuda.get_device_name(0)}"); plt.legend(); plt.grid(alpha=.3)
plt.savefig("softmax_benchmark.png", dpi=150, bbox_inches="tight"); plt.show()

## 3b. Wide rows: fused vs online vs PyTorch

Rows of 16k–131k columns (vocabulary-sized logits). The online kernel reads the input twice, so it moves 1.5× the bytes of the fused kernel; GB/s below is *effective* bandwidth (read once + write once) so all three are directly comparable as speed.

In [ ]:
rows = 256
wide = []
for n_cols in [16384, 32768, 65536, 131072]:
    x = torch.randn(rows, n_cols, device="cuda")
    t_online = triton.testing.do_bench(lambda: online_softmax(x))
    t_torch = triton.testing.do_bench(lambda: torch.softmax(x, dim=1))
    try:
        t_fused = triton.testing.do_bench(lambda: triton_softmax(x))
    except Exception as e:
        t_fused = float("nan"); print(f"fused kernel failed at {n_cols} cols: {type(e).__name__}")
    wide.append((n_cols, gbps(t_online, x), gbps(t_fused, x) if t_fused == t_fused else float("nan"),
                 gbps(t_torch, x), t_fused / t_online))

print(f"{'cols':>7} | {'online GB/s':>11} | {'fused GB/s':>10} | {'torch GB/s':>10} | {'online vs fused':>15}")
for c, on, fu, to, sp in wide:
    print(f"{c:>7} | {on:>11.1f} | {fu:>10.1f} | {to:>10.1f} | {sp:>14.2f}x")

## 4. Profiling

The PyTorch profiler shows how many GPU kernels each version launches and where the time goes. The naive version launches several kernels per call; the fused version launches one.

In [ ]:
from torch.profiler import profile, ProfilerActivity
x = torch.randn(4096, 4096, device="cuda")
for name, fn in [("naive", naive_softmax), ("triton", triton_softmax)]:
    for _ in range(3): fn(x)                      # warm-up / compile
    torch.cuda.synchronize()
    with profile(activities=[ProfilerActivity.CUDA]) as prof:
        for _ in range(10): fn(x)
        torch.cuda.synchronize()
    print(f"===== {name} =====")
    print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=8))

## 5. Summary for your resume

In [ ]:
best = max(results, key=lambda r: r[4])
mid  = [r for r in results if r[0] == 4096][0]
w = wide[-1]
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Fused @4096x4096: {mid[1]:.0f} GB/s | naive {mid[3]:.0f} GB/s ({mid[4]:.2f}x) | torch.softmax {mid[2]:.0f} GB/s ({mid[5]:.2f}x)")
print(f"Fused best speedup over naive: {best[4]:.2f}x at {best[0]} columns")
print(f"Online @{w[0]} cols: {w[1]:.0f} GB/s | fused {w[2]:.0f} GB/s | torch {w[3]:.0f} GB/s | online vs fused {w[4]:.2f}x")